### Embed + Export

#### Output
<div dir="rtl">
برای هر فایل ورودی، یک فایل خروجی با همون اسم در پوشه‌ی `EMBED_OUT_DIR` ساخته می‌شه، شامل فقط دو ستون: `id`, `embedding`. اگه فایلی رو قبلاً پردازش کرده باشید، دوباره پردازش نمی‌شه (resume خودکار بر اساس وجود فایل خروجی).
</div>

#### Package Installation

In [2]:
!pip install -q -U sentence-transformers tqdm pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00


#### Imports

In [9]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

import gc
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer


Mounted at /content/drive


#### Config

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
USE_E5_PREFIXES = True
EMBED_BATCH_SIZE = 1024
MAX_SEQ_LEN = 128

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Cleaned_data")
COMMENTS_DIR = BASE_DIR / "absa_results_comments"

EMBED_OUT_DIR = BASE_DIR / "embeddings_export"
(EMBED_OUT_DIR / "comments").mkdir(parents=True, exist_ok=True)

COMMENT_ID_COL = "id"
COMMENT_TEXT_COL = "raw_text_normalized"

device: cuda


#### Load Embedding Model

In [20]:
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LEN
EMBED_DIM = model.get_embedding_dimension()
print("embedding dim:", EMBED_DIM)


def embed_texts(texts, is_query=False):
    if USE_E5_PREFIXES:
        prefix = "query: " if is_query else "passage: "
        texts = [prefix + (t or "") for t in texts]

    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        vecs = model.encode(
            texts,
            batch_size=EMBED_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    return vecs.astype(np.float16)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedding dim: 768


#### Export Comments

In [23]:
def export_comments():
    files = sorted(COMMENTS_DIR.glob("*.parquet"))
    print(f"Found {len(files)} comment files")

    for file in files:
        out_path = EMBED_OUT_DIR / "comments" / file.name
        if out_path.exists():
            print(f"[comments] skip (already exported): {file.name}")
            continue

        df = pd.read_parquet(file, columns=[COMMENT_ID_COL, COMMENT_TEXT_COL])
        n = len(df)
        n_batches = math.ceil(n / EMBED_BATCH_SIZE)

        all_ids = []
        all_embeddings = []

        for b in tqdm(range(n_batches), desc=f"comments:{file.name}"):
            chunk = df.iloc[b * EMBED_BATCH_SIZE : (b + 1) * EMBED_BATCH_SIZE]
            ids = chunk[COMMENT_ID_COL].astype(str).tolist()
            texts = chunk[COMMENT_TEXT_COL].fillna("").astype(str).tolist()

            embeddings = embed_texts(texts, is_query=False)
            all_ids.extend(ids)
            all_embeddings.extend(embeddings.tolist())

        out_df = pd.DataFrame({"id": all_ids, "embedding": all_embeddings})
        out_df.to_parquet(out_path, index=False, engine="pyarrow")
        print(f"Saved -> {out_path}  ({len(out_df)} rows)")

        del df, out_df, all_ids, all_embeddings
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    print("\n✅ Comments export finished.")


export_comments()


Found 124 comment files
[comments] skip (already exported): absa_part_0000.parquet
[comments] skip (already exported): absa_part_0001.parquet
[comments] skip (already exported): absa_part_0002.parquet
[comments] skip (already exported): absa_part_0003.parquet
[comments] skip (already exported): absa_part_0004.parquet
[comments] skip (already exported): absa_part_0005.parquet
[comments] skip (already exported): absa_part_0006.parquet
[comments] skip (already exported): absa_part_0007.parquet
[comments] skip (already exported): absa_part_0008.parquet
[comments] skip (already exported): absa_part_0009.parquet
[comments] skip (already exported): absa_part_0010.parquet
[comments] skip (already exported): absa_part_0011.parquet
[comments] skip (already exported): absa_part_0012.parquet
[comments] skip (already exported): absa_part_0013.parquet
[comments] skip (already exported): absa_part_0014.parquet
[comments] skip (already exported): absa_part_0015.parquet
[comments] skip (already exporte

comments:absa_part_0069.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0069.parquet  (50000 rows)


comments:absa_part_0070.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0070.parquet  (50000 rows)
[comments] skip (already exported): absa_part_0071.parquet
[comments] skip (already exported): absa_part_0072.parquet
[comments] skip (already exported): absa_part_0073.parquet
[comments] skip (already exported): absa_part_0074.parquet
[comments] skip (already exported): absa_part_0075.parquet
[comments] skip (already exported): absa_part_0076.parquet
[comments] skip (already exported): absa_part_0077.parquet
[comments] skip (already exported): absa_part_0078.parquet
[comments] skip (already exported): absa_part_0079.parquet
[comments] skip (already exported): absa_part_0080.parquet
[comments] skip (already exported): absa_part_0081.parquet
[comments] skip (already exported): absa_part_0082.parquet
[comments] skip (already exported): absa_part_0083.parquet
[comments] skip (already exported): absa_part_0084.parquet
[comments] skip (already exported): absa_part_008

comments:absa_part_0094.parquet:   0%|          | 0/49 [00:00<?, ?it/s]

Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_data/embeddings_export/comments/absa_part_0094.parquet  (50000 rows)
[comments] skip (already exported): absa_part_0095.parquet
[comments] skip (already exported): absa_part_0096.parquet
[comments] skip (already exported): absa_part_0097.parquet
[comments] skip (already exported): absa_part_0098.parquet
[comments] skip (already exported): absa_part_0099.parquet
[comments] skip (already exported): absa_part_0100.parquet
[comments] skip (already exported): absa_part_0101.parquet
[comments] skip (already exported): absa_part_0102.parquet
[comments] skip (already exported): absa_part_0103.parquet
[comments] skip (already exported): absa_part_0104.parquet
[comments] skip (already exported): absa_part_0105.parquet
[comments] skip (already exported): absa_part_0106.parquet
[comments] skip (already exported): absa_part_0107.parquet
[comments] skip (already exported): absa_part_0108.parquet
[comments] skip (already exported): absa_part_010